# 03 — Modelagem e Avaliação
**CRISP-DM — etapas 4 e 5.** Treinamos 5 modelos com a mesma interface (perfil → ranking de jogos), ajustamos
hiperparâmetros num conjunto de **validação** e reportamos o resultado num conjunto de **teste** separado.

| Modelo | Tipo | Por que entra |
|---|---|---|
| Popularidade | baseline | qualquer modelo precisa superar "recomendar o mais popular" |
| Conteúdo TF-IDF (média / máximo) | baseado em conteúdo | proposta original do seminário |
| Colaborativo item-item | filtragem colaborativa | usa o comportamento real dos usuários (base B) |
| Fatoração SVD | fatoração de matriz | captura padrões latentes de gosto |
| Híbrido | conteúdo + colaborativo | une as duas fontes de informação |

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, carregamento as c, preparacao as p, graficos as g

config.criar_pastas()
g.aplicar_estilo()
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("Percentual de usuários amostrados:", config.PERCENTUAL_USUARIOS, "%")

import time
import joblib
from src import modelos as m, avaliacao as av

In [ ]:
catalogo = pd.read_parquet(config.ARQ_CATALOGO)
treino = pd.read_parquet(config.ARQ_TREINO)
teste = pd.read_parquet(config.ARQ_TESTE)

indice = m.IndiceCatalogo(catalogo)
matriz_ui = m.matriz_usuario_item(treino, indice)
casos = av.montar_casos(treino, teste, indice, n_usuarios=config.N_USUARIOS_AVALIACAO)
validacao, casos_teste = av.separar_validacao(casos)

densidade = matriz_ui.nnz / (matriz_ui.shape[0] * matriz_ui.shape[1])
print(f"Matriz usuário x jogo: {matriz_ui.shape} | densidade {densidade:.4%}")
print(f"Usuários avaliados: {len(casos):,} (validação {len(validacao):,} | teste {len(casos_teste):,})")

## Treinamento

In [ ]:
tempos = {}
def treinar(nome, fabrica):
    t = time.time()
    modelo = fabrica()
    tempos[nome] = time.time() - t
    print(f"{nome:<32} treinado em {tempos[nome]:.1f}s")
    return modelo

pop = treinar("Popularidade", lambda: m.Popularidade(indice, treino))
tfidf_media = treinar("Conteúdo TF-IDF (média)", lambda: m.ConteudoTFIDF(indice, "media"))
tfidf_max = treinar("Conteúdo TF-IDF (máximo)", lambda: m.ConteudoTFIDF(indice, "maximo"))
item_item = treinar("Colaborativo item-item", lambda: m.ColaborativoItemItem(indice, matriz_ui))
print(f"Vocabulário TF-IDF: {len(tfidf_media.vetorizador.vocabulary_):,} tokens")

## Ajuste de hiperparâmetros na validação (perfil de 5 jogos)

### SVD — número de dimensões latentes

In [ ]:
grade_svd = []
for d in [16, 32, 64, 128]:
    if d >= min(matriz_ui.shape):
        continue
    modelo = m.FatoracaoSVD(indice, matriz_ui, n_componentes=d)
    _, r = av.avaliar(modelo, validacao, tamanho_perfil=5)
    grade_svd.append({"dimensoes": d, "ndcg@10": r["ndcg@10"]})
grade_svd = pd.DataFrame(grade_svd)
melhor_d = int(grade_svd.loc[grade_svd["ndcg@10"].idxmax(), "dimensoes"])
svd = treinar("Fatoração SVD", lambda: m.FatoracaoSVD(indice, matriz_ui, n_componentes=melhor_d))
grade_svd

### Híbrido — peso α do conteúdo (α = 0 → só colaborativo; α = 1 → só conteúdo)

In [ ]:
grade_alpha = []
for alpha in np.round(np.arange(0, 1.01, 0.1), 1):
    h = m.Hibrido(indice, tfidf_media, item_item, peso_conteudo=alpha)
    _, r = av.avaliar(h, validacao, tamanho_perfil=5)
    grade_alpha.append({"alpha": alpha, "ndcg@10": r["ndcg@10"], "hit_rate@10": r["hit_rate@10"]})
grade_alpha = pd.DataFrame(grade_alpha)
melhor_alpha = float(grade_alpha.loc[grade_alpha["ndcg@10"].idxmax(), "alpha"])
hibrido = m.Hibrido(indice, tfidf_media, item_item, peso_conteudo=melhor_alpha)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(grade_alpha["alpha"], grade_alpha["ndcg@10"], color=g.AZUL, marker="o", markersize=6,
        markeredgecolor="white", markeredgewidth=1.5)
ax.scatter([melhor_alpha], [grade_alpha["ndcg@10"].max()], s=90, facecolor="none", edgecolor=g.TEXTO, zorder=4)
ax.annotate(f"melhor α = {melhor_alpha:.1f}", (melhor_alpha, grade_alpha["ndcg@10"].max()),
            xytext=(10, -18), textcoords="offset points", color=g.TEXTO)
ax.set_title("Figura 8 — Ajuste do peso do conteúdo no híbrido (validação)")
ax.set_xlabel("α (peso do conteúdo)")
ax.set_ylabel("NDCG@10")
g.salvar(fig, "fig08_ajuste_alpha_hibrido")
plt.show()
grade_alpha

## Avaliação final no conjunto de teste — 1 jogo × 5 jogos

In [ ]:
modelos = [pop, tfidf_media, tfidf_max, item_item, svd, hibrido]
resultados, por_usuario = [], {}
for modelo in modelos:
    for tam in (1, 5):
        u, r = av.avaliar(modelo, casos_teste, tamanho_perfil=tam, popularidade=pop.scores)
        r["ndcg_ic95"] = av.ic_bootstrap(u["ndcg"])
        resultados.append(r)
        por_usuario[(modelo.nome, tam)] = u

tabela = pd.DataFrame(resultados)
tabela.to_csv(config.RESULTADOS / "resultados_teste.csv", index=False)
tabela.drop(columns=["filtro_qualidade", "usuarios"]).set_index(["modelo", "perfil"])

In [ ]:
t = tabela.copy()
t["ic_inf"] = t["ndcg_ic95"].map(lambda x: x[0])
t["ic_sup"] = t["ndcg_ic95"].map(lambda x: x[1])
nomes = [mdl.nome for mdl in modelos]
y = np.arange(len(nomes))
altura = 0.38

fig, ax = plt.subplots(figsize=(9, 5))
for i, (perfil, cor) in enumerate([("1 jogo", g.AZUL), ("5 jogos", g.LARANJA)]):
    d = t[t["perfil"] == perfil].set_index("modelo").loc[nomes]
    pos = y + (i - 0.5) * altura
    ax.barh(pos, d["ndcg@10"], height=altura - 0.04, color=cor, label=f"Perfil com {perfil}")
    ax.errorbar(d["ndcg@10"], pos, xerr=[d["ndcg@10"] - d["ic_inf"], d["ic_sup"] - d["ndcg@10"]],
                fmt="none", ecolor=g.TEXTO_SECUNDARIO, elinewidth=1, capsize=2)
    for v, sup, yy in zip(d["ndcg@10"], d["ic_sup"], pos):
        ax.text(sup, yy, "  " + g.br(v, "{:.3f}"), va="center", fontsize=8.5, color=g.TEXTO_SECUNDARIO)
ax.set_yticks(y, nomes)
ax.invert_yaxis()
ax.margins(x=0.18)
ax.set_xlabel("NDCG@10 (barras de erro: IC 95% por bootstrap)")
g.eixo_br(ax, "x", "{:.2f}")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
ax.set_title("Figura 9 — Desempenho no teste: perfil de 1 jogo × 5 jogos", pad=28)
ax.grid(axis="y", visible=False)
g.salvar(fig, "fig09_ndcg_1x5_modelos")
plt.show()

### Teste estatístico pareado (Wilcoxon) — a diferença entre 1 e 5 jogos é significativa?

In [ ]:
testes = []
for modelo in modelos:
    if modelo is pop:
        continue  # popularidade ignora o perfil: 1 e 5 jogos são idênticos
    a = por_usuario[(modelo.nome, 1)]["ndcg"]
    b_ = por_usuario[(modelo.nome, 5)]["ndcg"]
    testes.append({"modelo": modelo.nome, "ndcg_1_jogo": a.mean(), "ndcg_5_jogos": b_.mean(),
                   **av.comparar_pareado(a, b_)})
testes = pd.DataFrame(testes)
testes["significativo_5%"] = testes["p_valor"] < 0.05
testes.to_csv(config.RESULTADOS / "teste_wilcoxon_1x5.csv", index=False)
testes

### Quantos jogos recentes usar? (curva de 1 a 5 jogos no perfil)

In [ ]:
curva = []
for modelo in [tfidf_media, item_item, hibrido]:
    for tam in range(1, 6):
        _, r = av.avaliar(modelo, casos_teste, tamanho_perfil=tam)
        curva.append({"modelo": modelo.nome, "jogos_no_perfil": tam, "ndcg@10": r["ndcg@10"]})
curva = pd.DataFrame(curva)
curva.to_csv(config.RESULTADOS / "curva_tamanho_perfil.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.2))
for cor, (nome, d) in zip(g.CORES, curva.groupby("modelo", sort=False)):
    ax.plot(d["jogos_no_perfil"], d["ndcg@10"], color=cor, marker="o", markersize=6,
            markeredgecolor="white", markeredgewidth=1.5, label=nome)
ax.set_xticks(range(1, 6))
ax.set_xlim(0.8, 5.2)
ax.set_xlabel("Jogos recentes usados no perfil")
ax.set_ylabel("NDCG@10")
ax.set_title("Figura 10 — Efeito do tamanho do perfil no desempenho (teste)")
ax.legend(loc="best")
g.salvar(fig, "fig10_curva_tamanho_perfil")
plt.show()

### Filtro de qualidade: ajuda ou atrapalha?

In [ ]:
abl = []
for modelo in [tfidf_media, item_item, hibrido]:
    for filtro in (False, True):
        _, r = av.avaliar(modelo, casos_teste, tamanho_perfil=5, filtro_qualidade=filtro, popularidade=pop.scores)
        abl.append(r)
ablacao = pd.DataFrame(abl)[["modelo", "filtro_qualidade", "precision@10", "recall@10",
                             "hit_rate@10", "ndcg@10", "cobertura_catalogo", "popularidade_media_recs"]]
ablacao.to_csv(config.RESULTADOS / "ablacao_filtro_qualidade.csv", index=False)
ablacao

## Avaliação qualitativa (proposta do seminário: notas 0, 1 e 2)
Perfis montados à mão com jogos conhecidos. O CSV exportado tem uma coluna `nota_relevancia` em branco para o grupo
preencher (2 = relevante, 1 = parcialmente relevante, 0 = irrelevante). Jogos que não existirem no catálogo são ignorados.

In [ ]:
perfis_teste = {
    "Seminário (FPS + RPG + Souls)": ["Call of Duty", "Cyberpunk 2077", "ELDEN RING", "Counter-Strike", "Red Dead Redemption 2"],
    "Indie / relaxante": ["Stardew Valley", "Terraria", "Hades", "Hollow Knight", "Celeste"],
    "Estratégia": ["Civilization VI", "Total War", "Crusader Kings", "Stellaris", "Age of Empires"],
}

def resolver(nomes):
    ids = []
    for nome in nomes:
        achados = indice.buscar_por_nome(nome)
        if len(achados):
            ids.append(int(achados.iloc[0]["app_id"]))
        else:
            print(f"  aviso: '{nome}' não encontrado no catálogo")
    return ids

linhas = []
for perfil_nome, nomes_jogos in perfis_teste.items():
    ids = resolver(nomes_jogos)
    if not ids:
        continue
    for tam in (1, 5):
        entrada = ids[-tam:]
        recs_q = hibrido.recomendar(entrada, k=10, filtro_qualidade=True)
        for pos_, linha in enumerate(recs_q.itertuples(), 1):
            linhas.append({"perfil": perfil_nome, "jogos_no_perfil": tam,
                           "entrada": " | ".join(indice.catalogo.set_index("app_id").loc[entrada, "nome"]),
                           "posicao": pos_, "recomendado": linha.nome, "score": round(linha.score, 4),
                           "nota_relevancia": ""})
qualitativa = pd.DataFrame(linhas)
qualitativa.to_csv(config.RESULTADOS / "avaliacao_qualitativa_para_preencher.csv", index=False)
qualitativa.head(20)

## Salvando o modelo final para a API (CRISP-DM — etapa 6, Implantação)

In [ ]:
joblib.dump({"indice": indice, "modelo": hibrido, "alpha": melhor_alpha, "svd_dimensoes": melhor_d},
            config.MODELOS / "recomendador.joblib", compress=3)
resumo_modelagem = {"melhor_alpha": melhor_alpha, "svd_dimensoes": melhor_d,
                    "tempos_treino_s": tempos, "densidade_matriz": densidade,
                    "usuarios_validacao": len(validacao), "usuarios_teste": len(casos_teste)}
with open(config.RESULTADOS / "resumo_modelagem.json", "w", encoding="utf-8") as f:
    json.dump(resumo_modelagem, f, ensure_ascii=False, indent=2, default=float)
resumo_modelagem